In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
import os
import glob

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("Датасет распакован!")
print("Файлы и папки:")
print(os.listdir('/content/dataset'))

In [ ]:
yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)

print("Найденные YAML-файлы:")
for file in yaml_files:
    print(file)

In [ ]:
import yaml

data_yaml_path = yaml_files[0]

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Содержимое data.yaml:")
print(data_config)

In [ ]:
import os

for root, dirs, files_list in os.walk('/content/dataset'):
    images = [
        f for f in files_list
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if images:
        print(root, "—", len(images), "изображений")

In [ ]:
DATA_YAML = yaml_files[0]

print("Путь к датасету:")
print(DATA_YAML)

In [ ]:
print("DATA_YAML =", DATA_YAML)

print("\nНазвания классов:")
print(data_config.get("names"))

print("\nКоличество классов:")
print(data_config.get("nc"))

print("\nПути к изображениям:")
print("train:", data_config.get("train"))
print("val:", data_config.get("val"))
print("test:", data_config.get("test"))

In [ ]:
import os
import glob

dataset_root = os.path.dirname(DATA_YAML)

train_dir = os.path.normpath(os.path.join(dataset_root, data_config["train"]))
val_dir = os.path.normpath(os.path.join(dataset_root, data_config["val"]))
test_dir = os.path.normpath(os.path.join(dataset_root, data_config["test"]))

def count_images(folder):
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    files = []

    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))

    return len(files)

print("TRAIN:", train_dir)
print("Количество:", count_images(train_dir))

print("\nVALIDATION:", val_dir)
print("Количество:", count_images(val_dir))

print("\nTEST:", test_dir)
print("Количество:", count_images(test_dir))

In [ ]:
import os
import glob

print("Ищем изображения во всём датасете...\n")

all_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    all_images.extend(
        glob.glob("/content/dataset/**/" + ext, recursive=True)
    )

print("ВСЕГО НАЙДЕНО ИЗОБРАЖЕНИЙ:", len(all_images))

print("\nПервые 10 найденных файлов:")
for img in all_images[:10]:
    print(img)

In [ ]:
import os
import glob

folders = [
    "/content/dataset/train/images",
    "/content/dataset/valid/images",
    "/content/dataset/test/images"
]

for folder in folders:
    images = []

    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        images.extend(glob.glob(os.path.join(folder, ext)))

    print("=" * 50)
    print(folder)
    print("Количество изображений:", len(images))

In [ ]:
import glob
import os

for folder in [
    "/content/dataset/train/labels",
    "/content/dataset/valid/labels",
    "/content/dataset/test/labels"
]:
    labels = glob.glob(os.path.join(folder, "*.txt"))

    print("=" * 50)
    print(folder)
    print("Количество файлов разметки:", len(labels))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random
import glob

train_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    train_images.extend(
        glob.glob("/content/dataset/train/images/" + ext)
    )

print("Изображений для обучения:", len(train_images))

sample_images = random.sample(
    train_images,
    min(6, len(train_images))
)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, image_path in zip(axes, sample_images):
    img = Image.open(image_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(image_path))
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import glob
import os

for folder in [
    "/content/dataset/train/labels",
    "/content/dataset/valid/labels",
    "/content/dataset/test/labels"
]:
    labels = glob.glob(os.path.join(folder, "*.txt"))

    print("=" * 50)
    print(folder)
    print("Количество файлов разметки:", len(labels))

In [ ]:
label_files = glob.glob("/content/dataset/train/labels/*.txt")

print("Всего файлов разметки:", len(label_files))

if len(label_files) > 0:
    print("\nПример разметки:")
    print("-" * 50)

    with open(label_files[0], "r") as f:
        print(f.read())

In [ ]:
print("Путь к YAML:")
print(DATA_YAML)

print("\nСодержимое YAML:")
with open(DATA_YAML, "r") as f:
    print(f.read())

In [ ]:
from collections import Counter
import glob
import os

counter = Counter()

label_files = glob.glob("/content/dataset/train/labels/*.txt")

for file in label_files:
    with open(file, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                class_id = line.split()[0]
                counter[class_id] += 1

print("Количество размеченных объектов:")
print(counter)

print("\nРасшифровка:")
print("0 =", data_config["names"][0])
print("1 =", data_config["names"][1])

In [ ]:
from ultralytics import YOLO

# Загружаем маленькую модель
model = YOLO("yolov8n.pt")

print("Модель YOLOv8n успешно загружена!")

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/helmet_models",
    name="yolov8n"
)

In [ ]:
from ultralytics import YOLO
import glob
import os

# Загружаем лучшую версию обученной модели
model_yolov8n = YOLO(
    "/content/helmet_models/yolov8n/weights/best.pt"
)

# Берём несколько изображений из test
test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(
        glob.glob("/content/dataset/test/images/" + ext)
    )

print("Тестовых изображений найдено:", len(test_images))

# Проверяем первые 6 изображений
sample = test_images[:6]

results = model_yolov8n.predict(
    source=sample,
    conf=0.25,
    save=True,
    project="/content/helmet_predictions",
    name="yolov8n_test",
    exist_ok=True
)

print("\nГотово!")
print("Результаты сохранены в:")
print("/content/helmet_predictions/yolov8n_test")

In [ ]:
total_helmet = 0
total_no_helmet = 0

for result in results:
    if result.boxes is None:
        continue

    for cls in result.boxes.cls:
        class_id = int(cls)

        if class_id == 0:
            total_helmet += 1
        elif class_id == 1:
            total_no_helmet += 1

print("СТАТИСТИКА")
print("=" * 30)
print("Обнаружено в каске:", total_helmet)
print("Обнаружено без каски:", total_no_helmet)
print("Всего обнаружений:", total_helmet + total_no_helmet)

In [ ]:
from IPython.display import display
from PIL import Image
import glob
import os

result_images = []

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    result_images.extend(
        glob.glob("/content/helmet_predictions/yolov8n_test/" + ext)
    )

print("Найдено обработанных изображений:", len(result_images))

for path in result_images:
    print(os.path.basename(path))
    display(Image.open(path))

In [ ]:
from ultralytics import YOLO

# Загружаем модель YOLOv8s
model = YOLO("yolov8s.pt")

# Обучаем на нашем датасете
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/helmet_models",
    name="yolov8s"
)

In [ ]:
from ultralytics import YOLO

# Загружаем YOLOv8m
model = YOLO("yolov8m.pt")

# Обучение
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=4,
    patience=10,
    project="/content/helmet_models",
    name="yolov8m"
)

In [ ]:
from ultralytics import YOLO

# Загружаем YOLOv9c
model = YOLO("yolov9c.pt")

# Обучение
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=4,
    patience=10,
    project="/content/helmet_models",
    name="yolov9c"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov9c.pt")

In [ ]:
from ultralytics import YOLO

# Загружаем YOLO11n
model = YOLO("yolo11n.pt")

print("Модель YOLO11n успешно загружена")

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/helmet_models",
    name="yolo11n"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo12n.pt")

print("Модель YOLO12n успешно загружена")

In [ ]:
from ultralytics import YOLO

# Загружаем YOLO12n
model = YOLO("yolo12n.pt")

# Обучаем на нашем датасете
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/helmet_models",
    name="yolo12n"
)

In [ ]:
from ultralytics import YOLO
import glob
import os

# Загружаем лучшую модель
model = YOLO(
    "/content/helmet_models/yolov8n/weights/best.pt"
)

# Все тестовые изображения
test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(
        glob.glob("/content/dataset/test/images/" + ext)
    )

print("Тестовых изображений:", len(test_images))

# Полное тестирование
results = model.predict(
    source=test_images,
    conf=0.25,
    save=True,
    project="/content/helmet_predictions",
    name="final_yolov8n",
    exist_ok=True
)

print("\nТестирование завершено!")
print("Результаты сохранены в:")
print("/content/helmet_predictions/final_yolov8n")

In [ ]:
from ultralytics import YOLO
import glob
import os
import shutil

# Загружаем лучшую модель
model = YOLO(
    "/content/helmet_models/yolov8n/weights/best.pt"
)

# Все тестовые изображения
test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(
        glob.glob("/content/dataset/test/images/" + ext)
    )

print("Всего тестовых изображений:", len(test_images))

# Папки для результатов
violation_dir = "/content/helmet_violations"
os.makedirs(violation_dir, exist_ok=True)

# Счётчики
total_helmet = 0
total_no_helmet = 0
images_with_violations = 0
images_without_detections = 0

# Обработка изображений
for image_path in test_images:

    results = model.predict(
        source=image_path,
        conf=0.25,
        verbose=False
    )

    result = results[0]

    has_no_helmet = False

    if result.boxes is not None:

        for cls in result.boxes.cls:

            class_id = int(cls)

            if class_id == 0:
                total_helmet += 1

            elif class_id == 1:
                total_no_helmet += 1
                has_no_helmet = True

    # Если обнаружен человек без каски —
    # сохраняем обработанное изображение
    if has_no_helmet:

        images_with_violations += 1

        # Получаем изображение с нанесёнными рамками
        annotated = result.plot()

        filename = os.path.basename(image_path)
        output_path = os.path.join(
            violation_dir,
            filename
        )

        from PIL import Image
        Image.fromarray(annotated[:, :, ::-1]).save(output_path)

    # Если вообще ничего не обнаружено
    if result.boxes is None or len(result.boxes) == 0:
        images_without_detections += 1


# Общая статистика
total_detections = total_helmet + total_no_helmet

if total_detections > 0:
    violation_percent = (
        total_no_helmet / total_detections * 100
    )
else:
    violation_percent = 0


print()
print("=" * 45)
print("ИТОГОВАЯ СТАТИСТИКА")
print("=" * 45)
print("Тестовых изображений:", len(test_images))
print("Обнаружено в каске:", total_helmet)
print("Обнаружено без каски:", total_no_helmet)
print("Всего обнаружений:", total_detections)
print("Изображений с нарушениями:", images_with_violations)
print("Изображений без обнаружений:", images_without_detections)
print(
    "Доля обнаружений без каски: "
    f"{violation_percent:.2f}%"
)
print("=" * 45)

print()
print("Изображения с нарушениями сохранены в:")
print(violation_dir)

In [ ]:
from PIL import Image
import glob
import os
from IPython.display import display

# Получаем сохранённые изображения с нарушениями
violation_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    violation_images.extend(
        glob.glob("/content/helmet_violations/" + ext)
    )

print("Сохранено изображений с нарушениями:", len(violation_images))

# Показываем первые 10 изображений
print("\nПервые изображения с обнаруженным отсутствием каски:\n")

for path in violation_images[:10]:
    print(os.path.basename(path))
    display(Image.open(path))

In [ ]:
from ultralytics import YOLO
import glob
import os
from PIL import Image
from IPython.display import display

model = YOLO("/content/helmet_models/yolov8n/weights/best.pt")

test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(
        glob.glob("/content/dataset/test/images/" + ext)
    )

print("Всего тестовых изображений:", len(test_images))

# Берём несколько изображений для визуального анализа
sample = test_images[:20]

results = model.predict(
    source=sample,
    conf=0.25,
    verbose=False
)

print("\nПримеры результатов:\n")

for i, result in enumerate(results):
    path = sample[i]

    print("=" * 60)
    print("Файл:", os.path.basename(path))

    if result.boxes is None or len(result.boxes) == 0:
        print("Объекты не обнаружены")
    else:
        for cls, conf in zip(
            result.boxes.cls.tolist(),
            result.boxes.conf.tolist()
        ):
            class_name = model.names[int(cls)]
            print(
                f"Обнаружено: {class_name}, "
                f"уверенность: {conf:.2f}"
            )

    # Показываем изображение с результатами
    annotated = result.plot()
    display(Image.fromarray(annotated[:, :, ::-1]))

In [ ]:
import glob
import os
from PIL import Image
from IPython.display import display

# Берём первые 20 изображений обучающей выборки
train_images = glob.glob("/content/dataset/train/images/*")

print("Всего изображений:", len(train_images))
print("\nПоказываю первые 20 изображений:\n")

for path in train_images[:20]:
    print(os.path.basename(path))
    display(Image.open(path))

In [ ]:
from ultralytics import YOLO

# Загружаем небольшую модель YOLO
model = YOLO("yolo11n.pt")

# Обращение к YAML автоматически скачает Construction-PPE
results = model.train(
    data="construction-ppe.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    project="/content/construction_ppe_test",
    name="download_test"
)

In [ ]:
import os
import glob
from PIL import Image
from IPython.display import display

# Ищем изображения Construction-PPE
search_paths = [
    "/root/.config/Ultralytics/datasets/construction-ppe",
    "/content/datasets/construction-ppe",
    "/content/construction-ppe"
]

dataset_path = None

for path in search_paths:
    if os.path.exists(path):
        dataset_path = path
        break

print("Найденный датасет:", dataset_path)

# Если путь отличается — ищем автоматически
if dataset_path is None:
    matches = []
    for root, dirs, files in os.walk("/root/.config/Ultralytics/datasets"):
        if "construction-ppe" in root.lower():
            matches.append(root)

    print("Найденные папки:", matches)

    if matches:
        dataset_path = matches[0]

# Ищем изображения
images = []

if dataset_path:
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        images.extend(
            glob.glob(dataset_path + "/**/" + ext, recursive=True)
        )

print("Всего найдено изображений:", len(images))

# Показываем первые 10
print("\nПервые изображения Construction-PPE:\n")

for path in images[:10]:
    print(os.path.basename(path))
    display(Image.open(path))

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="construction-ppe.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/construction_ppe_models",
    name="yolo11n"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="construction-ppe.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/construction_ppe_models",
    name="yolov8n"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="construction-ppe.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/construction_ppe_models",
    name="yolov8s"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="construction-ppe.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/construction_ppe_models",
    name="yolov8m"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo12n.pt")

results = model.train(
    data="construction-ppe.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    project="/content/construction_ppe_models",
    name="yolo12n"
)

In [ ]:
from ultralytics import YOLO

# Загружаем лучшую модель
model = YOLO("/content/construction_ppe_models/yolov8m/weights/best.pt")

# Проверяем модель на тестовой выборке
results = model.val(
    data="construction-ppe.yaml",
    split="test",
    imgsz=640
)

In [ ]:
from ultralytics import YOLO
import os

# Загружаем лучшую модель
model = YOLO("/content/construction_ppe_models/yolov8m/weights/best.pt")

# Путь к тестовым изображениям
test_images = "/content/datasets/construction-ppe/images/test"

# Папка для сохранения результатов
save_dir = "/content/helmet_violations_final"

# Запускаем детектирование
results = model.predict(
    source=test_images,
    imgsz=640,
    conf=0.25,
    save=True,
    project=save_dir,
    name="detections"
)

print("Готово!")
print("Результаты сохранены в:", save_dir + "/detections")

In [ ]:
from ultralytics import YOLO
import os
import shutil
from collections import Counter

# Загружаем обученную модель
model = YOLO("/content/construction_ppe_models/yolov8m/weights/best.pt")

# Тестовые изображения
test_images = "/content/datasets/construction-ppe/images/test"

# Папка для нарушений
violations_dir = "/content/helmet_violations_final/no_helmet_images"
os.makedirs(violations_dir, exist_ok=True)

# Получаем список изображений
image_files = [
    os.path.join(test_images, f)
    for f in os.listdir(test_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Статистика
total_images = len(image_files)
images_with_violations = 0
total_no_helmet = 0
total_helmet = 0
total_person = 0

violation_files = []

# Обрабатываем изображения
for image_path in image_files:

    results = model.predict(
        source=image_path,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    result = results[0]

    # Считаем классы
    classes = result.boxes.cls.cpu().numpy().astype(int)

    no_helmet_count = sum(classes == 7)
    helmet_count = sum(classes == 0)
    person_count = sum(classes == 6)

    total_no_helmet += no_helmet_count
    total_helmet += helmet_count
    total_person += person_count

    # Если есть нарушение — сохраняем изображение
    if no_helmet_count > 0:
        images_with_violations += 1
        violation_files.append(os.path.basename(image_path))

        # Сохраняем исходное изображение
        shutil.copy2(
            image_path,
            os.path.join(
                violations_dir,
                os.path.basename(image_path)
            )
        )

# Процент изображений с нарушениями
violation_percent = (
    images_with_violations / total_images * 100
    if total_images > 0 else 0
)

print("=" * 60)
print("СТАТИСТИКА КОНТРОЛЯ НАЛИЧИЯ КАСКИ")
print("=" * 60)

print(f"Всего тестовых изображений: {total_images}")
print(f"Изображений с нарушениями: {images_with_violations}")
print(f"Доля изображений с нарушениями: {violation_percent:.2f}%")
print(f"Всего обнаружено касок: {total_helmet}")
print(f"Всего обнаружено людей: {total_person}")
print(f"Всего обнаружено случаев 'без каски': {total_no_helmet}")

print("=" * 60)
print("Изображения с нарушениями сохранены в:")
print(violations_dir)
print("=" * 60)

print("\nФайлы с нарушениями:")
for filename in violation_files:
    print(filename)

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

violations_dir = "/content/helmet_violations_final/no_helmet_images"

files = [
    os.path.join(violations_dir, f)
    for f in os.listdir(violations_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Количество изображений с нарушениями:", len(files))
print("\nПервые изображения:")

for f in sorted(files)[:6]:
    print(os.path.basename(f))

# Показываем первые 6 изображений
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, path in zip(axes.ravel(), sorted(files)[:6]):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(os.path.basename(path))
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO
import os
import shutil

model = YOLO("/content/construction_ppe_models/yolov8m/weights/best.pt")

test_images = "/content/datasets/construction-ppe/images/test"
normal_dir = "/content/helmet_violations_final/normal_helmet_images"

os.makedirs(normal_dir, exist_ok=True)

normal_files = []

image_files = [
    os.path.join(test_images, f)
    for f in os.listdir(test_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for image_path in image_files:
    results = model.predict(
        source=image_path,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    result = results[0]
    classes = result.boxes.cls.cpu().numpy().astype(int)

    helmet_count = sum(classes == 0)
    no_helmet_count = sum(classes == 7)

    # Сохраняем изображения, где обнаружена каска
    # и нет обнаружения no_helmet
    if helmet_count > 0 and no_helmet_count == 0:
        normal_files.append(os.path.basename(image_path))
        shutil.copy2(
            image_path,
            os.path.join(normal_dir, os.path.basename(image_path))
        )

print("=" * 60)
print("ПРОВЕРКА ИЗОБРАЖЕНИЙ БЕЗ НАРУШЕНИЙ")
print("=" * 60)
print(f"Всего тестовых изображений: {len(image_files)}")
print(f"Изображений с обнаруженной каской без no_helmet: {len(normal_files)}")
print()
print("Изображения сохранены в:")
print(normal_dir)
print("=" * 60)

print("\nПримеры:")
for filename in normal_files[:15]:
    print(filename)

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("/content/construction_ppe_models/yolov8m/weights/best.pt")

test_images = "/content/datasets/construction-ppe/images/test"

# Папка с примерами результатов
examples_dir = "/content/helmet_violations_final/examples"
os.makedirs(examples_dir, exist_ok=True)

# Берём несколько изображений с нарушениями
example_files = [
    "image1205.jpg",
    "image1145.jpg",
    "image1354.jpg",
    "image1256.jpg"
]

for filename in example_files:
    image_path = os.path.join(test_images, filename)

    if os.path.exists(image_path):
        model.predict(
            source=image_path,
            imgsz=640,
            conf=0.25,
            save=True,
            project=examples_dir,
            name="annotated",
            exist_ok=True,
            verbose=False
        )

print("=" * 60)
print("ГОТОВО")
print("=" * 60)
print("Аннотированные изображения сохранены в:")
print("/content/helmet_violations_final/examples/annotated")